In [ ]:
import os
import torch
import numpy as np
os.environ["KERAS_BACKEND"] = "torch"

In [ ]:
if torch.backends.mps.is_available():
    print("Apple's MPS backend (used by PyTorch on M1/M2 Macs) does not support float64 (double precision).")
    mps_enabled = True
    torch.set_default_dtype(torch.float32)
    print("set default to float32")

In [ ]:
import keras
import tensorflow as tf

In [ ]:
if keras.backend.backend() != "torch":
    print(f"warning: keras backend is set to {keras.backend.backend()}, restart jupyter kernel!!!!")
    raise RuntimeError()

In [ ]:
keras.backend.backend()

In [ ]:
import json

global config
with open('./config/keras_nn.json') as keras_nn_config:
    config = json.load(keras_nn_config)
    print("config loaded")

In [ ]:
# Root-level fields
batch_size = config["batchSize"]
scaler_enabled = config["scaler"]["enabled"]
scaler_type = config["scaler"]["type"]

seed = config["seed"]

validation_enabled = config["validation"]["enabled"]
validation_ratio = config["validation"]["ratio"]

input_size = config["nInputs"]
output_size = config["nTargets"]


In [ ]:
keras.utils.set_random_seed(seed)

In [ ]:
%load_ext tensorboard
# now available at http://localhost:6006/?

In [ ]:
# Dataset initialization

from utils.data_loader import get_ml_cup_data
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, MaxAbsScaler

def _scaler():
    if not scaler_enabled:
        return None

    match scaler_type:
        case "Standard":
            return StandardScaler()
        case "MinMax":
            return MinMaxScaler()
        case "Robust":
            return RobustScaler()
        case "MaxAbsScaler":
            return MaxAbsScaler()
        case _:
            return None

train_loader, test_loader = get_ml_cup_data(
    batch_size, 
    scaler=_scaler(),
    mps=mps_enabled
    )

In [ ]:
train_loader.dataset.X.shape, train_loader.dataset.y.shape

In [ ]:
test_loader.dataset.X.shape, test_loader.dataset.y.shape

In [ ]:
import numpy as np

y_mean = train_loader.dataset.y.mean(axis=0)        # (4,)

y_pred_baseline = np.tile(y_mean, (len(train_loader.dataset.y), 1))

mee_errors = np.linalg.norm(train_loader.dataset.y - y_pred_baseline, axis=1)
mse_errors = np.square(train_loader.dataset.y - y_pred_baseline)

mee_baseline = mee_errors.mean()
mse_baseline = mse_errors.mean()

print("Baseline MEE:", mee_baseline)
print("Baseline MSE:", mse_baseline)

In [ ]:
# --- Neural network ("nn") section ---
nn_optimizer = config["nn"]["optimizer"]
nn_epochs = config["nn"]["epochs"]

# Hidden layers
nn_hidden: list[dict] = config["nn"]["hidden"]

In [ ]:
from keras import Sequential
from keras.layers import Input, Dense

In [ ]:
from losses import MeanEuclidianError

mee = MeanEuclidianError(name="mee", dtype=torch.float32)

In [ ]:
def build_model(hp):
    model = Sequential()
    model.add(Input(shape=(input_size,)))

    # --- Layer 1 units ---
    lambda_1 = hp.Choice(
        "lambda_1", [1e-5, 1e-4]
    )
    model.add(Dense(
        8,
        activation="leaky_relu",
        kernel_regularizer=keras.regularizers.l2(lambda_1),
        kernel_initializer=keras.initializers.GlorotNormal(seed=seed)
    ))

    # --- Layer 2 units ---
    lambda_2 = hp.Choice(
        "lambda_2", [1e-4, 1e-3]
    )
    model.add(Dense(
        4,
        activation="leaky_relu",
        kernel_regularizer=keras.regularizers.l2(lambda_2),
        kernel_initializer=keras.initializers.GlorotNormal(seed=seed)
    ))
    
    model.add(Dense(output_size))
    lr = hp.Choice("lr", [1e-4, 3e-4, 1e-3, 3e-3])
    momentum = 0.9
    optimizer = keras.optimizers.SGD(learning_rate=lr, momentum=momentum, nesterov=True)
    model.compile(optimizer=optimizer, loss="mse", metrics=[mee])
    return model

In [ ]:
from keras_tuner import HyperParameters

build_model(HyperParameters()).summary()

In [ ]:
from keras.callbacks import EarlyStopping

early_stopping_cb = EarlyStopping(
    patience=15,
    mode="min",
    restore_best_weights=True,
    baseline=mse_baseline,
    verbose=1
)

In [ ]:
import utils.keras as ukeras

In [ ]:
from sklearn.model_selection import KFold
from torch.utils.data import Subset, DataLoader
from keras_tuner import GridSearch

train_dataset = train_loader.dataset
n_samples = len(train_dataset)
all_indices = np.arange(n_samples)

k = 5

saved_keras_models_root = f"keras/models"
if os.path.isdir(saved_keras_models_root) and len(os.listdir(saved_keras_models_root)) > 0:
    # saved_keras_models_root is not empty, so we already saved the models
    pass
else:
    kf = KFold(n_splits=k, shuffle=True, random_state=seed)

    fold_results = []
    fold_best_hparams = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(all_indices)):
        print(f"\n=== Fold {fold + 1}/{k} ===")
        train_indices = all_indices[train_idx]
        val_indices = all_indices[val_idx]
        
        # indices for this fold
        train_indices = all_indices[train_idx]
        val_indices = all_indices[val_idx]

        train_subset = Subset(train_dataset, train_indices)
        val_subset = Subset(train_dataset, val_indices)

        train_loader_fold = DataLoader(
            train_subset,
            batch_size=batch_size,
            shuffle=True
        )

        val_loader_fold = DataLoader(
            val_subset,
            batch_size=batch_size,
            shuffle=False
        )

        tuner = GridSearch(
            hypermodel=build_model,
            objective="val_loss",
            directory="keras",
            project_name=f"sgd_nn_fold{fold+1}",
            max_trials=15,
            max_consecutive_failed_trials=5,
            seed=seed,
            overwrite=True,
        )

        hyperparameters_path = f"keras/models/hyperparameters/fold{fold+1}"
        
        tuner.search(
            train_loader_fold,
            validation_data=val_loader_fold,
            epochs=100,
            verbose=1,
            callbacks=[early_stopping_cb]
        )
        ukeras.save_hyperparameters(tuner, dir=hyperparameters_path)

        best_trial = tuner.oracle.get_best_trials(num_trials=1)[0]
        best_val_loss = best_trial.metrics.get_last_value("val_loss")
        fold_results.append(best_val_loss)

        print(f"Best val_loss for fold {fold + 1}: {best_val_loss}")

In [ ]:
fold_results

In [ ]:
fold_results_path = "keras/fold_results.json"
if os.path.exists(fold_results_path):
    with open(fold_results_path, "r") as f:
        fold_results = json.load(f)
else:
    with open(fold_results_path, "w+") as f:
        json.dump(fold_results, f, indent=2)

In [ ]:
fold_results

In [ ]:
best_fold_idx = int(np.argmin(fold_results))
print("best_fold_idx", best_fold_idx)
best_hp_overall = ukeras.load_hyperparameters(f"keras/models/hyperparameters/fold{best_fold_idx+1}", hp_filename="best_hp_config.json")
print("best_hp_overall", best_hp_overall.values)

In [ ]:
from keras.callbacks import TensorBoard
import datetime

tensorboard_cb = TensorBoard(
    histogram_freq=1,
    write_graph=True,
    write_images=False
)

In [ ]:
saving_best_model_root = "keras/models/best"

if os.path.exists(saving_best_model_root):
    for name in os.listdir(saving_best_model_root):
        path = os.path.join(saving_best_model_root, name)
        if os.path.isfile(path):
            saving_best_model_root += "/" + name
    best_model = ukeras.load_saved_model(saving_best_model_root)
    print("model loaded")
else:
    best_model = build_model(best_hp_overall)
    model_name = ukeras.dict_to_filename(hyperparams=best_hp_overall, prefix=f"best_model_8x4")
    tensorboard_cb.log_dir = ukeras.log_dir(model_name)
    best_model.fit(
        x=train_dataset.X,
        y=train_dataset.y,
        batch_size=batch_size,
        epochs=150,
        verbose=1,
        validation_split=.10,
        callbacks=[tensorboard_cb, early_stopping_cb]
    )
    os.makedirs(saving_best_model_root, exist_ok=True)
    best_model.save(saving_best_model_root + "/" + model_name + ".keras")

In [ ]:
save_ensemble_models_basepath = "keras/models/ensemble"
hyperparameters_path_prefix = "keras/models/hyperparameters/fold"

In [ ]:
best_hparams = []
for fold in range(1, k+1):
    fold_k_best_hp = ukeras.load_hyperparameters(hyperparameters_path_prefix + str(fold), hp_filename="best_hp_config.json")
    best_hparams.append(fold_k_best_hp)

In [ ]:
ensemble_models = []
id = 0
for i, hparams in enumerate(best_hparams):
    print(f"hyperparameters={hparams.values}")
    bootstrap_indices = np.random.choice(n_samples, size=n_samples, replace=True)
    X_boot = train_dataset.X[bootstrap_indices]
    y_boot = train_dataset.y[bootstrap_indices]

    m = build_model(hparams)
    model_name = f"{ukeras.dict_to_filename(hparams, prefix="sgd_fold_8x4")}"
    tensorboard_cb.log_dir = ukeras.log_dir(model_name)
    saving_ensemble_models_path = f"{save_ensemble_models_basepath}/{model_name}_{id}.keras"
    id += 1
    if os.path.exists(saving_ensemble_models_path):
        ensemble_models.append(ukeras.load_saved_model(saving_ensemble_models_path))
        print(f"loading {len(ensemble_models)} models...")
    else:
        m.fit(
            x=X_boot,
            y=y_boot,
            batch_size=batch_size,
            epochs=150,
            verbose=1,
            validation_split=.10,
            callbacks=[tensorboard_cb, early_stopping_cb]
        )
        os.makedirs(save_ensemble_models_basepath, exist_ok=True)
        ensemble_models.append(m)
        m.save(saving_ensemble_models_path)

In [ ]:
len(ensemble_models)

In [ ]:
%tensorboard --logdir logs/fit

In [ ]:
def ensemble_predict(models, X):
    preds = [m.predict(X) for m in models]
    return np.mean(preds, axis=0)

In [ ]:
y_pred_single = best_model.predict(test_loader.dataset.X)
y_pred_ensemble = ensemble_predict(ensemble_models, test_loader.dataset.X)

In [ ]:
mee_single = mee.call(test_loader.dataset.y, y_pred_single)
mee_ensemble = mee.call(test_loader.dataset.y, y_pred_ensemble)

mse_single = np.square(test_loader.dataset.y - y_pred_single).mean()
mse_ensemble = np.square(test_loader.dataset.y - y_pred_ensemble).mean()

tr_mee_single = mee.call(train_dataset.y, best_model.predict(train_dataset.X))
tr_mee_ensemble = mee.call(train_dataset.y, ensemble_predict(ensemble_models, train_dataset.X))

tr_mse_single = np.square(train_dataset.y - best_model.predict(train_dataset.X)).mean()
tr_mse_ensemble = np.square(train_dataset.y - ensemble_predict(ensemble_models, train_dataset.X)).mean()

In [ ]:
errors_tr = ukeras.build_results_json(
    tr_mee_single, 
    tr_mee_ensemble, 
    mee_baseline, 
    tr_mse_single, 
    tr_mse_ensemble, 
    mse_baseline
)

In [ ]:
errors_tr

In [ ]:
errors_ts = ukeras.build_results_json(
    mee_single, 
    mee_ensemble, 
    mee_baseline, 
    mse_single, 
    mse_ensemble, 
    mse_baseline,
    prefix="ts",
    print_baseline=True
)

In [ ]:
errors_ts

In [ ]:
with open("keras/assessment.json", "w+") as fp:
    print("writing assessment.json")
    json.dump({**errors_tr, **errors_ts}, fp, indent=2)

In [ ]:
if tr_mee_single < mee_single:
    print(f"overfitting mee_single {tr_mee_single - mee_single}")

if tr_mee_ensemble < mee_ensemble:
    print(f"overfitting mee_ensemble {tr_mee_ensemble - mee_ensemble}")

if tr_mse_single < mse_single:
    print(f"overfitting mse_single {tr_mee_single - mse_single}")

if tr_mse_ensemble < mse_ensemble:
    print(f"overfitting mse_ensemble {tr_mse_ensemble - mse_ensemble}")

In [ ]:
tuner.results_summary()

In [ ]:
best_hps = tuner.get_best_hyperparameters(10)
model = build_model(best_hps[0])

In [ ]:
history = model.fit(
        train_loader,
        epochs=1,
        verbose=1
    )

In [ ]:
model.summary()

In [ ]:
from keras.callbacks import EarlyStopping, TensorBoard
import datetime

def log_dir(name, append:str=None):
    BASE = f"logs/{name}/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    if append:
        BASE += "_" + append
    return BASE

In [ ]:
# Early stopping
nn_es_enabled = config["nn"]["earlyStopping"]["enabled"]                ## true/false
nn_es_patience = config["nn"]["earlyStopping"]["patience"]              ## int
nn_es_mode = config["nn"]["earlyStopping"]["mode"]                      ## min/max
nn_es_restore_best = config["nn"]["earlyStopping"]["restoreBestWeight"] ## true/false

In [ ]:
tensorboard_cb = TensorBoard(
    log_dir=log_dir("fit"),
    histogram_freq=1,
    write_graph=True,
    write_images=False
)

In [ ]:
tensorboard_cb.log_dir = log_dir("fit", nn_optimizer)

history = model.fit(
    train_loader, 
    validation_data=validation_loader, 
    epochs=nn_epochs,
    callbacks=[tensorboard_cb]
    )

In [ ]:
%tensorboard --logdir logs/fit

In [ ]:
# evaluate model
results = model.evaluate(test_loader, return_dict=True)
print(results)

In [ ]:
# write logs to a separate folder
writer = tf.summary.create_file_writer(log_dir("eval", nn_optimizer))

with writer.as_default():
    for k, v in results.items():
        tf.summary.scalar(k, v, step=0)

writer.close()

In [ ]:
%tensorboard --logdir logs/eval

In [ ]:
# --- Optuna ("optuna") section ---
optuna_epochs = config["optuna"]["epochs"]

# Suggestions for hidden layers
optuna_hidden_1_range = config["optuna"]["suggestions"]["hidden"][0]["range"]
optuna_hidden_1_activation = config["optuna"]["suggestions"]["hidden"][0]["activation"]

optuna_hidden_2_range = config["optuna"]["suggestions"]["hidden"][1]["range"]
optuna_hidden_2_activation = config["optuna"]["suggestions"]["hidden"][1]["activation"]

# Optuna early stopping
optuna_es_enabled = config["optuna"]["earlyStopping"]["enabled"]
optuna_es_patience = config["optuna"]["earlyStopping"]["patience"]
optuna_es_mode = config["optuna"]["earlyStopping"]["mode"]
optuna_es_restore_best = config["optuna"]["earlyStopping"]["restoreBestWeight"]

In [ ]:
## Optuna
import optuna
import tensorflow as tf

def objective(trial):
    # Suggest hyperparameters
    units1 = trial.suggest_int("units1", 16, 128)
    units2 = trial.suggest_int("units2", 16, 128)
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-2, log=True)

    # Build model
    model = Sequential([
        Input(shape=(input_size,)),
        # Dense layers are fully connected layers
        Dense(units1, activation='relu'),
        Dense(units2, activation='relu'),
        Dense(output_size)
    ])

    optimizer = keras.optimizers.SGD(learning_rate=learning_rate)
    model.compile(
        optimizer=optimizer,
        loss=mee
    )

    tensorboard_cb.log_dir = log_dir("fit", f"OPTUNA_TRIAL#{trial.number}")

    pruning_cb = optuna.integration.KerasPruningCallback(trial, "val_loss")
    
    checkpoint_path = f"checkpoints/optuna_trial_{trial.number}.keras"

    checkpoint_cb = keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_path,
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        save_weights_only=False
    )
    
    # Train model
    history = model.fit(
        train_loader,
        validation_data=validation_loader,
        epochs=optuna_epochs,
        verbose=0,
        callbacks=[tensorboard_cb, pruning_cb, checkpoint_cb]
    )

    val_loss = history.history["val_loss"][-1]
    return val_loss

In [ ]:
from optuna.samplers import TPESampler

sampler = TPESampler(seed=seed)
study = optuna.create_study(sampler=sampler, direction="minimize")
study.optimize(objective, n_trials=30)

print("Best trial:", study.best_trial.params)

In [ ]:
from utils.optuna import delete_pruned_trial_dirs

In [ ]:
delete_pruned_trial_dirs(root_path="logs/fit", study=study)

In [ ]:
study.best_trial

In [ ]:
best_trial = study.best_trial
best_model_path = f"checkpoints/optuna_trial_{best_trial.number}.keras"
best_model = keras.models.load_model(best_model_path)

In [ ]:
test_loss = best_model.evaluate(test_loader)

In [ ]:
writer = tf.summary.create_file_writer(log_dir("eval", f"OPTUNA_TRIAL#{best_trial.number}"))
with writer.as_default():
    tf.summary.scalar("loss", test_loss, step=0)
    writer.flush()

In [ ]:
%tensorboard --logdir logs/eval